In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)


In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [3]:
#collecting data
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") #Sentinel-2 Surface Reflectance data

filtered_image = s2 \
    .filterBounds(region) \
    .sort('system:time_start')\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
    #.filterDate('2022-01-01', '2022-12-31') \
    

print(f"Number of images found: {filtered_image.size().getInfo()}")

Number of images found: 596


In [4]:
#now making sure they have full coverage of the region
strict_collection = filtered_image.filter(ee.Filter.contains(
        leftField='.geo', #.geo refers to the geometry of the image
        rightValue=region #the region we defined earlier 
    )
)

print(f"Number of images with full coverage: {strict_collection.size().getInfo()}")

Number of images with full coverage: 588


In [5]:
min_timestamp = strict_collection.aggregate_min("system:time_start")
max_timestamp = strict_collection.aggregate_max("system:time_start")
print(f"Min Timestamp: {min_timestamp.getInfo()}")
print(f"Max Timestamp: {max_timestamp.getInfo()}")
print('First Date:', ee.Date(min_timestamp).format('YYYY-MM-dd').getInfo())
print('Last Date:', ee.Date(max_timestamp).format('YYYY-MM-dd').getInfo())

Min Timestamp: 1453092113893
Max Timestamp: 1767156117904
First Date: 2016-01-18
Last Date: 2025-12-31


In [7]:

def create_monthly_medians(collection):
    start_date = ee.Date(collection.aggregate_min("system:time_start"))
    end_date = ee.Date(collection.aggregate_max("system:time_start"))
    print("start date: ", start_date.getInfo(), " end date: ", end_date.getInfo())
    
    months_diff = end_date.difference(start_date, 'month').round()
    print("months difference: ", months_diff.getInfo())
    
    month_list = ee.List.sequence(0, months_diff)
    print("month list: ", month_list.getInfo())
    
    # creating median composite for each month
    def monthly_median(month_offset):
        month_start = start_date.advance(month_offset, 'month')
        month_end = month_start.advance(1, 'month')
        
        monthly_collection = collection.filterDate(month_start, month_end)
        
        median_image = monthly_collection.median().set({
            'system:time_start': month_start.millis(),
            'month': month_start.format('YYYY-MM')
        })
        
        return median_image
    
    monthly_medians = month_list.map(monthly_median)
    return ee.ImageCollection(monthly_medians)

monthly_collection = create_monthly_medians(strict_collection)

print(f"number of monthly images: {monthly_collection.size().getInfo()}")


visualized_collection = monthly_collection.map(lambda img: img.visualize(
    bands=['B4', 'B3', 'B2'],
    min=0,
    max=3000
))

start date:  {'type': 'Date', 'value': 1453092113893}  end date:  {'type': 'Date', 'value': 1767156117904}
months difference:  119
month list:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119]
number of monthly images: 120


In [9]:

import datetime


now = datetime.datetime.now()
time_stamp = now.strftime("%Y_%m_%d_%H_%M_%S")

file_name = 'My_First_Satellite_Timelapse_' + time_stamp
print('Exporting file as:', file_name)

# Create the export task
task = ee.batch.Export.video.toDrive(
    collection=visualized_collection,
    folder='GEE_Exports',
    description=file_name,
    dimensions=720,    
    framesPerSecond=10, 
    region=region
)

task.start()


Exporting file as: My_First_Satellite_Timelapse_2026_01_03_14_24_17


In [10]:
try:
    while task.active():
        print('Polling for task (id: {task.id}). Current status: {task.status()}')
        print(task)
        import time
        time.sleep(30)
        
except KeyboardInterrupt:
    print('stopped')

Polling for task (id: {task.id}). Current status: {task.status()}
<Task N7WZEKNAY6RZZKEND5IAN5BD Type.EXPORT_VIDEO: My_First_Satellite_Timelapse_2026_01_03_14_24_17 (State.UNSUBMITTED)>
Polling for task (id: {task.id}). Current status: {task.status()}
<Task N7WZEKNAY6RZZKEND5IAN5BD Type.EXPORT_VIDEO: My_First_Satellite_Timelapse_2026_01_03_14_24_17 (State.UNSUBMITTED)>
Polling for task (id: {task.id}). Current status: {task.status()}
<Task N7WZEKNAY6RZZKEND5IAN5BD Type.EXPORT_VIDEO: My_First_Satellite_Timelapse_2026_01_03_14_24_17 (State.UNSUBMITTED)>
Polling for task (id: {task.id}). Current status: {task.status()}
<Task N7WZEKNAY6RZZKEND5IAN5BD Type.EXPORT_VIDEO: My_First_Satellite_Timelapse_2026_01_03_14_24_17 (State.UNSUBMITTED)>
Polling for task (id: {task.id}). Current status: {task.status()}
<Task N7WZEKNAY6RZZKEND5IAN5BD Type.EXPORT_VIDEO: My_First_Satellite_Timelapse_2026_01_03_14_24_17 (State.UNSUBMITTED)>
Polling for task (id: {task.id}). Current status: {task.status()}
<Tas